In [1]:
# type: ignore


import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


load_dotenv()

llm_model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME"),
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),
    extra_body={"enable_thinking": False},
)

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""

    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


structured_llm_grader = llm_model.with_structured_output(GradeDocuments)

system_prompt = """You are a grader assessing relevance of a retrieved document to a user question. \n 
    If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question.
    
    You MUST output a valid JSON object with ONLY the key "binary_score"."""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

retrieval_grader = grade_prompt | structured_llm_grader

In [11]:
document = """Agentic RAG（Agentic Retrieval-Augmented Generation，智能体增强检索生成）
是一种将 自主 AI 智能体（Autonomous Agents）与 检索增强生成（RAG）相结合的先进架构。
它超越了传统 RAG 的“一次检索 + 一次生成”静态流程，赋予系统动态规划、多步推理、自我反思和工具调用等能力。"""

retrieval_grader.invoke({"document": document, "question": "什么是 Agentic RAG?"})

GradeDocuments(binary_score='yes')

In [12]:
document = """Agent（智能体）是指能够感知环境、自主决策并采取行动以实现特定目标的软件实体。
它不仅仅是被动响应指令的程序，而是具备主动性、目标导向性和一定程度的自治能力的系统。"""

retrieval_grader.invoke({"document": document, "question": "什么是 Agentic RAG?"})

GradeDocuments(binary_score='no')